# 00 Archive-Listing Manifest, Cleaning, and Split

Input Kaggle datasets:

- `/kaggle/input/datasets/tundng111/vimed-pet-ct-part1-raw-no-autoextract-v3`
- `/kaggle/input/datasets/tundng111/vimed-pet-ct-part2-raw-no-autoextract-v1`
- `/kaggle/input/datasets/tundng111/vimed-pet-ct-part3-raw-no-autoextract-v1`

This notebook builds the Report/Q1 manifest without extracting the full CT/PET raw
volumes. It reads split ZIP inventories with `7z l -slt`, extracts only report JSON
files for text parsing, and optionally extracts a small CT/PET sample per source base
for shape checking and visual preview.

Expected year-level counts from the ViMed-PET paper/card:

- 2017: 215 studies
- 2018: 462 studies
- 2019: 339 studies
- 2023: 1,741 studies
- Total: 2,757 studies


In [ ]:
from pathlib import Path, PurePosixPath
import os, re, json, shutil, subprocess, hashlib, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

warnings.filterwarnings("ignore")

OUTPUT_ROOT = Path("/kaggle/working/vimedpet_q1_outputs") if Path("/kaggle/working").exists() else Path("vimedpet_q1_outputs")
MANIFEST_DIR = OUTPUT_ROOT / "00_manifest"
FIGURES_DIR = MANIFEST_DIR / "figures"
TABLES_DIR = MANIFEST_DIR / "tables"
SCRATCH_ROOT = Path("/kaggle/temp/vimedpet_q1_listing") if Path("/kaggle").exists() else OUTPUT_ROOT / "_scratch_listing"
JSON_ROOT = SCRATCH_ROOT / "report_json"
STAGE_ROOT = SCRATCH_ROOT / "archive_stage"
SAMPLE_ROOT = SCRATCH_ROOT / "sample_npy"

for p in [MANIFEST_DIR, FIGURES_DIR, TABLES_DIR]:
    p.mkdir(parents=True, exist_ok=True)
if SCRATCH_ROOT.exists():
    shutil.rmtree(SCRATCH_ROOT, ignore_errors=True)
for p in [SCRATCH_ROOT, JSON_ROOT, STAGE_ROOT, SAMPLE_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

SOURCE_BASES = {
    "Part 1": ["PETCT_2017", "PETCT_2018", "PETCT_2019"],
    "Part 2": ["PETCT_2023_1", "PETCT_2023_2"],
    "Part 3": ["PETCT_2023_3"],
}
SOURCE_TO_PART = {src: part for part, items in SOURCE_BASES.items() for src in items}
SOURCES = [src for items in SOURCE_BASES.values() for src in items]

EXPECTED_TOTAL_CASES = 2757
EXPECTED_YEAR_COUNTS = {
    "PETCT_2017": 215,
    "PETCT_2018": 462,
    "PETCT_2019": 339,
    "PETCT_2023": 1741,
}
SPLIT_RATIOS = (0.70, 0.15, 0.15)
SPLIT_SEED = 391

# RUN_ARCHIVE_TEST=True runs CRC testing with 7z t. It uses little disk
# but can take a long time, so keep False for normal manifest runs.
RUN_ARCHIVE_TEST = False
EXTRACT_REPORT_JSON = True
SAMPLE_NPY_CHECK_PER_SOURCE = 1
MAKE_SAMPLE_PREVIEW = True

INPUT_HINTS = [
    "/kaggle/input/datasets/tundng111/vimed-pet-ct-part1-raw-no-autoextract-v3",
    "/kaggle/input/datasets/tundng111/vimed-pet-ct-part2-raw-no-autoextract-v1",
    "/kaggle/input/datasets/tundng111/vimed-pet-ct-part3-raw-no-autoextract-v1",
    "/kaggle/input",
    str(Path.cwd()),
]

def show_disk_usage(label):
    print(f"\nDisk usage: {label}")
    for raw_path in ["/kaggle/temp", "/tmp", "/kaggle/working"]:
        p = Path(raw_path)
        if p.exists():
            usage = shutil.disk_usage(p)
            print(
                raw_path,
                "free GiB:", round(usage.free / 1024**3, 2),
                "used GiB:", round(usage.used / 1024**3, 2),
                "total GiB:", round(usage.total / 1024**3, 2),
            )

print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("SCRATCH_ROOT:", SCRATCH_ROOT)
show_disk_usage("after setup")


In [ ]:
def existing_roots():
    roots = []
    for x in INPUT_HINTS:
        p = Path(x)
        if p.exists():
            roots.append(p)
    seen, out = set(), []
    for r in roots:
        rr = r.resolve()
        if rr not in seen:
            out.append(rr)
            seen.add(rr)
    return out

def sevenzip_binary():
    for exe in ["7z", "7zz", "7za"]:
        try:
            subprocess.run([exe], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
            return exe
        except Exception:
            continue
    raise RuntimeError("7z/7zz/7za not found in this runtime")

def find_archive_group(source_base):
    hits = []
    for root in existing_roots():
        for marker in list(root.rglob(f"{source_base}.zipraw")) + list(root.rglob(f"{source_base}.zip")):
            parts = sorted(marker.parent.glob(f"{source_base}.z[0-9][0-9]"))
            hits.append({"marker": marker, "parts": parts, "folder": marker.parent})
    if not hits:
        raise FileNotFoundError(f"Cannot find split ZIP marker for {source_base}")
    return sorted(hits, key=lambda h: len(h["parts"]), reverse=True)[0]

def validate_archive_group(source_base, group):
    parts = group["parts"]
    names = [p.name for p in parts]
    expected = [f"{source_base}.z{i:02d}" for i in range(1, len(parts) + 1)]
    missing = [name for name in expected if name not in names]
    print(f"\nArchive group for {source_base}")
    print("  marker:", group["marker"])
    print("  marker size GiB:", round(group["marker"].stat().st_size / 1024**3, 3))
    print("  split parts:", len(parts), names[:3], "..." if len(names) > 6 else "", names[-3:])
    print("  split parts size GiB:", round(sum(p.stat().st_size for p in parts) / 1024**3, 3))
    if missing:
        raise FileNotFoundError(f"Missing split volumes for {source_base}: {missing}")

def symlink_required(src, dst):
    if dst.exists() or dst.is_symlink():
        dst.unlink()
    try:
        os.symlink(src, dst)
    except Exception as exc:
        raise RuntimeError(
            f"Cannot create symlink {dst} -> {src}. This notebook avoids copying archives to protect temp disk."
        ) from exc

def stage_archive(source_base, group):
    stage = STAGE_ROOT / source_base
    if stage.exists():
        shutil.rmtree(stage, ignore_errors=True)
    stage.mkdir(parents=True, exist_ok=True)
    for part in group["parts"]:
        symlink_required(part, stage / part.name)
    # The final volume was uploaded as .zipraw. Stage it as .zip so 7z
    # recognizes the multi-volume ZIP set without copying the file.
    symlink_required(group["marker"], stage / f"{source_base}.zip")
    return stage / f"{source_base}.zip"

def norm_path(x):
    return str(x).replace("\\", "/").lstrip("./")

def tail_from_thang(path):
    path = norm_path(path)
    m = re.search(r"(THANG\s+\d+/.+)$", path, flags=re.I)
    return m.group(1) if m else path

def run_7z(args, tail_chars=1600):
    result = subprocess.run(args, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    if tail_chars:
        print(result.stdout[-tail_chars:])
    return result

def list_archive(source_base, zip_path):
    cmd = [sevenzip_binary(), "l", "-slt", str(zip_path)]
    result = run_7z(cmd, tail_chars=1200)
    if result.returncode != 0:
        raise RuntimeError(f"7z listing failed for {source_base}")
    entries, cur = [], {}
    for line in result.stdout.splitlines():
        if line.startswith("Path = "):
            if cur.get("Path"):
                entries.append(cur)
            cur = {"Path": line.split(" = ", 1)[1]}
        elif " = " in line and cur:
            k, v = line.split(" = ", 1)
            cur[k] = v
    if cur.get("Path"):
        entries.append(cur)
    rows = []
    for item in entries:
        p = norm_path(item.get("Path", ""))
        size = item.get("Size", "")
        if not p or not re.search(r"\.(npy|json)$", p, flags=re.I):
            continue
        if "THANG " not in p.upper():
            continue
        try:
            size_int = int(size)
        except Exception:
            size_int = 0
        rows.append({
            "source_part": source_base,
            "dataset_part": SOURCE_TO_PART[source_base],
            "archive_path": p,
            "archive_tail": tail_from_thang(p),
            "archive_tail_key": tail_from_thang(p).lower(),
            "size_bytes": size_int,
            "crc": item.get("CRC", ""),
        })
    return pd.DataFrame(rows)

def extract_json_reports(source_base, zip_path):
    out = JSON_ROOT / source_base
    if out.exists():
        shutil.rmtree(out, ignore_errors=True)
    out.mkdir(parents=True, exist_ok=True)
    if not EXTRACT_REPORT_JSON:
        return out
    cmd = [sevenzip_binary(), "x", str(zip_path), f"-o{out}", "-y", "-r", "*.json"]
    result = run_7z(cmd, tail_chars=1200)
    if result.returncode != 0:
        raise RuntimeError(f"JSON extraction failed for {source_base}")
    return out

def test_archive_crc(source_base, zip_path):
    if not RUN_ARCHIVE_TEST:
        return "not_run"
    cmd = [sevenzip_binary(), "t", str(zip_path)]
    result = run_7z(cmd, tail_chars=1600)
    return "ok" if result.returncode == 0 else f"failed_{result.returncode}"


In [ ]:
def parse_day_patient(name):
    m = re.search(r"day[_-]?(\d+).*patient[_-]?(\d+)", name, flags=re.I)
    if not m:
        return None, None
    return int(m.group(1)), str(int(m.group(2)))

def read_json(path):
    for enc in ["utf-8", "utf-8-sig"]:
        try:
            return json.loads(Path(path).read_text(encoding=enc))
        except Exception:
            pass
    return {}

def clean_text(x):
    if x is None:
        return ""
    s = str(x).replace("\r", " ").replace("\t", " ").strip()
    s = re.sub(r"[ \u00a0]+", " ", s)
    s = re.sub(r"\n{3,}", "\n\n", s)
    return s.strip()

def normalize_key_name(x):
    import unicodedata
    s = str(x or "")
    # Replace Vietnamese D-with-stroke before accent decomposition. The source uses
    # unicode escapes so this code remains safe even if an editor/log cannot render Vietnamese.
    s = s.replace("\u0111", "d").replace("\u0110", "D")
    s = unicodedata.normalize("NFKD", s)
    s = "".join(ch for ch in s if not unicodedata.combining(ch))
    s = s.lower()
    s = re.sub(r"[^a-z0-9]+", " ", s)
    return re.sub(r"\s+", " ", s).strip()

def get_first_normalized(obj, normalized_names):
    if not isinstance(obj, dict):
        return ""
    wanted = {normalize_key_name(x) for x in normalized_names}
    for key, value in obj.items():
        if normalize_key_name(key) in wanted:
            return value
    return ""

def nested_text(value):
    if value is None:
        return ""
    if isinstance(value, dict):
        parts = []
        for k, v in value.items():
            t = nested_text(v)
            if t:
                parts.append(f"{clean_text(k)}: {t}")
        return "\n".join(parts)
    if isinstance(value, list):
        return "\n".join(t for t in (nested_text(v) for v in value) if t)
    return clean_text(value)

def report_fields(report_path):
    obj = read_json(report_path)
    if not isinstance(obj, dict):
        raw = nested_text(obj)
        return raw, "", raw, "", "", "", ""

    findings_obj = get_first_normalized(obj, [
        "mo ta hinh anh", "mo ta", "findings", "ket qua"
    ])
    finding_parts = []
    if isinstance(findings_obj, dict):
        for section, value in findings_obj.items():
            value_text = nested_text(value)
            if value_text:
                finding_parts.append(f"{clean_text(section)}: {value_text}")
    else:
        value_text = nested_text(findings_obj)
        if value_text:
            finding_parts.append(value_text)
    findings = "\n".join(finding_parts)

    impression = clean_text(get_first_normalized(obj, [
        "nhan dinh ket qua", "ket luan", "impression", "conclusion"
    ]))
    indication = clean_text(get_first_normalized(obj, ["chi dinh", "indication"]))
    clinical_history = clean_text(get_first_normalized(obj, [
        "benh su lam sang", "lam sang", "clinical history", "clinical_history"
    ]))
    sex = clean_text(get_first_normalized(obj, ["gioi tinh", "sex", "gender"]))
    birth_year = clean_text(get_first_normalized(obj, ["nam sinh", "birth year", "birth_year"]))

    target_parts = []
    if findings:
        target_parts.append("FINDINGS:\n" + findings)
    if impression:
        target_parts.append("IMPRESSION:\n" + impression)
    report_text = "\n\n".join(target_parts).strip()
    if not report_text:
        report_text = nested_text(obj)
    return findings, impression, report_text, indication, clinical_history, sex, birth_year

def json_map_for_source(source_base):
    root = JSON_ROOT / source_base
    mapping = {}
    if not root.exists():
        return mapping
    for p in root.rglob("*.json"):
        tail = tail_from_thang(p.as_posix()).lower()
        mapping[tail] = p
    return mapping

def normalize_uint8(x, lo=None, hi=None):
    x = np.asarray(x, dtype=np.float32)
    if lo is None:
        lo = np.nanpercentile(x, 1)
    if hi is None:
        hi = np.nanpercentile(x, 99)
    x = np.clip(x, lo, hi)
    return ((x - lo) / max(hi - lo, 1e-6) * 255).astype(np.uint8)

def extract_selected_files(source_base, zip_path, archive_paths):
    out = SAMPLE_ROOT / source_base
    if out.exists():
        shutil.rmtree(out, ignore_errors=True)
    out.mkdir(parents=True, exist_ok=True)
    if not archive_paths:
        return out
    cmd = [sevenzip_binary(), "x", str(zip_path), f"-o{out}", "-y"] + archive_paths
    result = run_7z(cmd, tail_chars=1000)
    if result.returncode != 0:
        raise RuntimeError(f"Sample NPY extraction failed for {source_base}")
    return out


In [ ]:
all_frames = []
inventory_frames = []
archive_summary_rows = []
sample_rows = []
preview_items = []

for source_base in SOURCES:
    show_disk_usage(f"before listing {source_base}")
    group = find_archive_group(source_base)
    validate_archive_group(source_base, group)
    zip_path = stage_archive(source_base, group)
    try:
        inventory = list_archive(source_base, zip_path)
        inventory_frames.append(inventory)
        inventory.to_csv(TABLES_DIR / f"{source_base}_archive_inventory.csv", index=False, encoding="utf-8-sig")
        crc_status = test_archive_crc(source_base, zip_path)
        json_root = extract_json_reports(source_base, zip_path)
        json_by_tail = json_map_for_source(source_base)

        by_tail = {r.archive_tail_key: r for r in inventory.itertuples(index=False)}
        ct_inventory = inventory[
            inventory["archive_tail"].str.contains(r"/CT/", case=False, regex=True)
            & inventory["archive_tail"].str.endswith(".npy")
        ].copy()

        rows = []
        source_sample_candidates = []
        for item in ct_inventory.itertuples(index=False):
            ct_tail = item.archive_tail
            ct_key = item.archive_tail_key
            stem = PurePosixPath(ct_tail).stem
            pet_tail = re.sub(r"/CT/", "/PET/", ct_tail, flags=re.I)
            report_tail = re.sub(r"/CT/", "/reports/", ct_tail, flags=re.I)
            report_tail = re.sub(r"\.npy$", ".json", report_tail, flags=re.I)
            pet_key = pet_tail.lower()
            report_key = report_tail.lower()
            pet_item = by_tail.get(pet_key)
            report_item = by_tail.get(report_key)
            report_local = json_by_tail.get(report_key)

            findings = impression = report_text = indication = clinical_history = sex = birth_year = ""
            if report_local and report_local.exists():
                findings, impression, report_text, indication, clinical_history, sex, birth_year = report_fields(report_local)

            month_match = re.search(r"THANG\s+(\d+)", ct_tail, flags=re.I)
            month = int(month_match.group(1)) if month_match else None
            day, patient_id = parse_day_patient(stem)
            case_key = f"{source_base}|{ct_tail}"
            ct_size = int(item.size_bytes)
            pet_size = int(pet_item.size_bytes) if pet_item is not None else 0
            report_size = int(report_item.size_bytes) if report_item is not None else 0
            row = {
                "source_part": source_base,
                "dataset_part": SOURCE_TO_PART[source_base],
                "month": month,
                "day": day,
                "patient_id": patient_id,
                "global_patient_key": f"{source_base}|{patient_id}",
                "case_id": hashlib.sha1(case_key.encode()).hexdigest()[:12],
                "raw_case_key": case_key,
                "ct_img_path": ct_tail,
                "pet_img_path": pet_tail,
                "report_path": report_tail,
                "ct_path": item.archive_path,
                "pet_path": pet_item.archive_path if pet_item is not None else pet_tail,
                "report_path_resolved": report_item.archive_path if report_item is not None else report_tail,
                "ct_exists": True,
                "pet_exists": pet_item is not None,
                "report_exists": report_item is not None,
                "ct_size_bytes": ct_size,
                "pet_size_bytes": pet_size,
                "report_size_bytes": report_size,
                "ct_archive_entry_ok": ct_size > 128,
                "pet_archive_entry_ok": pet_size > 128,
                "report_archive_entry_ok": report_size > 0,
                "ct_readable": ct_size > 128,
                "pet_readable": pet_size > 128,
                "array_check_method": "archive_inventory_size_check; sampled_npy_load_check_saved_separately",
                "ct_error": "",
                "pet_error": "",
                "report_text_empty": str(report_text).strip() == "",
                "findings": findings,
                "impression": impression,
                "report_text_clean": report_text,
                "target_text": f"Findings:\n{findings}\n\nImpression:\n{impression}".strip(),
                "report_word_count": len(str(report_text).split()),
                "suv_mention_count": len(re.findall(r"SUV", str(report_text), flags=re.I)),
                "sex": sex,
                "birth_year": birth_year,
                "indication": indication,
                "clinical_history": clinical_history,
            }
            rows.append(row)
            if row["pet_exists"] and len(source_sample_candidates) < SAMPLE_NPY_CHECK_PER_SOURCE:
                source_sample_candidates.append(row)

        source_df = pd.DataFrame(rows)
        all_frames.append(source_df)
        archive_summary_rows.append({
            "source_part": source_base,
            "dataset_part": SOURCE_TO_PART[source_base],
            "archive_files_listed": int(len(inventory)),
            "ct_entries": int((inventory["archive_tail"].str.contains(r"/CT/", case=False, regex=True) & inventory["archive_tail"].str.endswith(".npy")).sum()),
            "pet_entries": int((inventory["archive_tail"].str.contains(r"/PET/", case=False, regex=True) & inventory["archive_tail"].str.endswith(".npy")).sum()),
            "json_entries": int(inventory["archive_tail"].str.endswith(".json").sum()),
            "cases_from_ct": int(len(source_df)),
            "crc_test_status": crc_status,
        })

        if source_sample_candidates:
            archive_paths = []
            for row in source_sample_candidates:
                archive_paths += [row["ct_path"], row["pet_path"]]
            sample_out = extract_selected_files(source_base, zip_path, archive_paths)
            for row in source_sample_candidates:
                try:
                    ct_file = sample_out / row["ct_path"]
                    pet_file = sample_out / row["pet_path"]
                    ct = np.load(ct_file, mmap_mode="r")
                    pet = np.load(pet_file, mmap_mode="r")
                    sample_rows.append({
                        "source_part": source_base,
                        "case_id": row["case_id"],
                        "ct_path": row["ct_path"],
                        "pet_path": row["pet_path"],
                        "ct_shape": str(tuple(ct.shape)),
                        "pet_shape": str(tuple(pet.shape)),
                        "ct_dtype": str(ct.dtype),
                        "pet_dtype": str(pet.dtype),
                        "sample_npy_read_ok": True,
                        "sample_error": "",
                    })
                    if MAKE_SAMPLE_PREVIEW:
                        ct_mid = np.asarray(ct[ct.shape[0] // 2])
                        pet_mip = np.max(np.asarray(pet), axis=0)
                        preview_items.append((source_base, row["case_id"], normalize_uint8(ct_mid, -1000, 1000), normalize_uint8(pet_mip)))
                except Exception as exc:
                    sample_rows.append({
                        "source_part": source_base,
                        "case_id": row["case_id"],
                        "ct_path": row["ct_path"],
                        "pet_path": row["pet_path"],
                        "ct_shape": "",
                        "pet_shape": "",
                        "ct_dtype": "",
                        "pet_dtype": "",
                        "sample_npy_read_ok": False,
                        "sample_error": str(exc),
                    })
            shutil.rmtree(SAMPLE_ROOT / source_base, ignore_errors=True)
    finally:
        shutil.rmtree(STAGE_ROOT / source_base, ignore_errors=True)
        shutil.rmtree(JSON_ROOT / source_base, ignore_errors=True)
        show_disk_usage(f"after cleanup {source_base}")

all_cases = pd.concat(all_frames, ignore_index=True)
all_inventory = pd.concat(inventory_frames, ignore_index=True)
archive_summary = pd.DataFrame(archive_summary_rows)
sample_checks = pd.DataFrame(sample_rows)


In [ ]:
all_cases = all_cases.drop_duplicates(subset=["source_part", "ct_img_path", "pet_img_path", "report_path"]).reset_index(drop=True)
all_cases["source_year"] = all_cases["source_part"].str.extract(r"(PETCT_\d{4})", expand=False).fillna(all_cases["source_part"])

source_counts = all_cases["source_part"].value_counts().sort_index().rename_axis("source_part").reset_index(name="all_cases")
year_counts = all_cases["source_year"].value_counts().sort_index().rename_axis("source_year").reset_index(name="all_cases")
expected_check = pd.DataFrame([
    {
        "source_year": k,
        "expected_cases": v,
        "computed_cases": int(year_counts.loc[year_counts.source_year == k, "all_cases"].sum()),
    }
    for k, v in EXPECTED_YEAR_COUNTS.items()
])
expected_check["status"] = np.where(expected_check["expected_cases"] == expected_check["computed_cases"], "OK", "CHECK")

issue_mask = (
    (~all_cases["ct_exists"]) | (~all_cases["pet_exists"]) | (~all_cases["report_exists"]) |
    (~all_cases["ct_archive_entry_ok"]) | (~all_cases["pet_archive_entry_ok"]) |
    (all_cases["report_text_empty"])
)
bad_records = all_cases[issue_mask].copy()
clean = all_cases[~issue_mask].copy().reset_index(drop=True)

def present_count(series):
    return int(series.fillna("").astype(str).str.strip().ne("").sum())

section_completeness = pd.DataFrame([
    {"field": "findings", "present": present_count(clean["findings"])},
    {"field": "impression", "present": present_count(clean["impression"])},
    {"field": "report_text_clean", "present": present_count(clean["report_text_clean"])},
])
section_completeness["total"] = int(len(clean))
section_completeness["coverage"] = section_completeness["present"] / section_completeness["total"].clip(lower=1)
section_completeness["coverage_percent"] = (section_completeness["coverage"] * 100).round(2)

def classify_issue(row):
    issues = []
    if not bool(row.get("ct_exists", False)): issues.append("missing_ct")
    if not bool(row.get("pet_exists", False)): issues.append("missing_pet")
    if not bool(row.get("report_exists", False)): issues.append("missing_report")
    if bool(row.get("ct_exists", False)) and not bool(row.get("ct_archive_entry_ok", False)): issues.append("ct_archive_size_error")
    if bool(row.get("pet_exists", False)) and not bool(row.get("pet_archive_entry_ok", False)): issues.append("pet_archive_size_error")
    if bool(row.get("report_text_empty", False)): issues.append("empty_report_text")
    return "|".join(issues) if issues else "ok"

if len(bad_records):
    bad_records["issue_type"] = bad_records.apply(classify_issue, axis=1)
else:
    bad_records["issue_type"] = []

rng = np.random.default_rng(SPLIT_SEED)
patients = np.array(sorted(clean["global_patient_key"].astype(str).unique()))
rng.shuffle(patients)
n = len(patients)
n_train = int(round(n * SPLIT_RATIOS[0]))
n_val = int(round(n * SPLIT_RATIOS[1]))
split_map = {}
for x in patients[:n_train]: split_map[x] = "train"
for x in patients[n_train:n_train+n_val]: split_map[x] = "validation"
for x in patients[n_train+n_val:]: split_map[x] = "test"
clean["clean_split"] = clean["global_patient_key"].map(split_map)
all_cases = all_cases.merge(clean[["case_id", "clean_split"]], on="case_id", how="left")
bad_records["clean_split"] = bad_records["global_patient_key"].map(split_map).fillna("excluded") if len(bad_records) else []

def overlap(df):
    sets = {s: set(df.loc[df.clean_split == s, "global_patient_key"].astype(str)) for s in ["train", "validation", "test"]}
    return {
        "train_validation": len(sets["train"] & sets["validation"]),
        "train_test": len(sets["train"] & sets["test"]),
        "validation_test": len(sets["validation"] & sets["test"]),
    }

summary = {
    "all_cases": int(len(all_cases)),
    "clean_cases": int(len(clean)),
    "bad_records": int(len(bad_records)),
    "expected_total_cases": int(EXPECTED_TOTAL_CASES),
    "matches_expected_total": bool(len(all_cases) == EXPECTED_TOTAL_CASES),
    "clean_matches_expected_total": bool(len(clean) == EXPECTED_TOTAL_CASES),
    "unique_patients": int(clean["global_patient_key"].nunique()),
    "split_counts": clean["clean_split"].value_counts().to_dict(),
    "patient_overlap": overlap(clean),
    "source_counts": clean["source_part"].value_counts().to_dict(),
    "year_counts": all_cases["source_year"].value_counts().to_dict(),
    "mean_report_words": float(clean["report_word_count"].mean()),
    "median_report_words": float(clean["report_word_count"].median()),
    "findings_present": int(section_completeness.loc[section_completeness.field == "findings", "present"].iloc[0]),
    "impression_present": int(section_completeness.loc[section_completeness.field == "impression", "present"].iloc[0]),
    "impression_coverage": float(section_completeness.loc[section_completeness.field == "impression", "coverage"].iloc[0]),
    "array_check_method": "archive inventory for all CT/PET files plus sampled np.load checks",
    "sample_npy_checks": int(len(sample_checks)),
    "sample_npy_read_failures": int((~sample_checks["sample_npy_read_ok"]).sum()) if len(sample_checks) else 0,
}

all_cases.to_csv(MANIFEST_DIR / "q1_all_cases_manifest.csv", index=False, encoding="utf-8-sig")
clean.to_csv(MANIFEST_DIR / "q1_clean_manifest.csv", index=False, encoding="utf-8-sig")
bad_records.to_csv(MANIFEST_DIR / "q1_bad_records.csv", index=False, encoding="utf-8-sig")
source_counts.to_csv(TABLES_DIR / "q1_source_counts.csv", index=False, encoding="utf-8-sig")
year_counts.to_csv(TABLES_DIR / "q1_year_counts.csv", index=False, encoding="utf-8-sig")
expected_check.to_csv(TABLES_DIR / "q1_expected_count_check.csv", index=False, encoding="utf-8-sig")
archive_summary.to_csv(TABLES_DIR / "q1_archive_summary.csv", index=False, encoding="utf-8-sig")
sample_checks.to_csv(TABLES_DIR / "q1_sample_npy_checks.csv", index=False, encoding="utf-8-sig")
section_completeness.to_csv(TABLES_DIR / "q1_section_completeness.csv", index=False, encoding="utf-8-sig")
(MANIFEST_DIR / "q1_manifest_summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")

display(Markdown("### Manifest summary"))
print(json.dumps(summary, ensure_ascii=False, indent=2))
display(Markdown("### Structured report section coverage"))
display(section_completeness)
display(Markdown("### Expected count check"))
display(expected_check)
display(Markdown("### Archive inventory summary"))
display(archive_summary)
display(Markdown("### Cases by source base"))
display(source_counts)
display(Markdown("### Cases by split"))
display(clean.groupby(["dataset_part", "source_part", "clean_split"]).size().rename("cases").reset_index())
display(Markdown("### Sample NPY load checks"))
display(sample_checks)
display(Markdown("### Bad records"))
bad_cols = ["source_part", "case_id", "issue_type", "ct_exists", "pet_exists", "report_exists", "ct_archive_entry_ok", "pet_archive_entry_ok", "report_text_empty", "ct_path", "pet_path", "report_path_resolved"]
display(bad_records[[c for c in bad_cols if c in bad_records.columns]].head(50))

assert summary["patient_overlap"] == {"train_validation": 0, "train_test": 0, "validation_test": 0}
assert len(all_cases) == EXPECTED_TOTAL_CASES, f"Expected {EXPECTED_TOTAL_CASES} all cases, got {len(all_cases)}"
assert (expected_check["status"] == "OK").all(), expected_check
assert summary["sample_npy_read_failures"] == 0, sample_checks
assert summary["findings_present"] / max(summary["clean_cases"], 1) >= 0.90, section_completeness
assert summary["impression_coverage"] >= 0.90, (
    "Impression coverage is unexpectedly low. This usually means the report JSON parser is wrong. "
    "The parser normalizes Vietnamese keys and expects normalized key 'nhan dinh ket qua'."
)
print("Saved manifest folder:", MANIFEST_DIR)


In [ ]:
if MAKE_SAMPLE_PREVIEW and preview_items:
    rows = len(preview_items)
    fig, axes = plt.subplots(rows, 2, figsize=(7.2, 2.5 * rows), constrained_layout=True)
    if rows == 1:
        axes = np.array([axes])
    for i, (source_base, case_id, ct_img, pet_img) in enumerate(preview_items):
        axes[i, 0].imshow(ct_img, cmap="gray")
        axes[i, 0].set_title(f"{source_base} CT\n{case_id}", fontsize=9)
        axes[i, 0].axis("off")
        axes[i, 1].imshow(pet_img, cmap="magma")
        axes[i, 1].set_title("PET MIP", fontsize=9)
        axes[i, 1].axis("off")
    preview_path = FIGURES_DIR / "q1_sample_ct_pet_preview.png"
    fig.savefig(preview_path, dpi=180, bbox_inches="tight")
    plt.show()
    print("Saved preview:", preview_path)
